In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# بارگذاری متغیرهای محیطی
load_dotenv()

# مقداردهی مدل جمینای با دمای صفر
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest", 
    temperature=0
)

print("✅ Gemini model initialized!")

✅ Gemini model initialized!


In [2]:
# تابع پارسر سفارشی (Custom Parser)
def parse_final_answer(text: str):
    # کلمه کلیدی که به مدل می‌گوییم جواب نهایی را بعد از آن بنویسد
    delimiter = "FINAL_ANSWER:"

    if delimiter in text:
        # متن را از محل جداکننده به دو قسمت تقسیم می‌کنیم
        parts = text.split(delimiter)
        reasoning = parts[0].strip()
        answer = parts[1].strip()

        return {
            "thought_process": reasoning,  # فکر را نگه می‌داریم
            "result": answer               # جواب نهایی
        }
    else:
        return {"error": "Could not find final answer", "raw": text}

print("✅ Custom Delimiter Parser function is ready!")

✅ Custom Delimiter Parser function is ready!


In [3]:
# پرامپت با دستورالعمل جداکننده
template = """
مسئله زیر را حل کن.
مهم: ابتدا مرحله به مرحله فکر کن (Reasoning).
سپس پاسخ نهایی را دقیقاً بعد از عبارت "FINAL_ANSWER:" بنویس.

مسئله: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# ساخت زنجیره
chain_delimiter = prompt | model | StrOutputParser() | RunnableLambda(parse_final_answer)

print("⏳ Executing Delimiter Strategy...\n")

# اجرا
output = chain_delimiter.invoke({"question": "اگر سن من (۲۰ سال) دو برابر برادرم باشد، وقتی من ۴۰ ساله شوم برادرم چند ساله است؟"})

if "error" in output:
    print(f"❌ خطا: {output['error']}")
    print(output['raw'])
else:
    print("--- 🧠 تفکرات مدل (برای لاگ سیستم) ---")
    print(output['thought_process'])
    
    print("\n--- ✅ پاسخ نهایی (برای نمایش به کاربر) ---")
    print(output['result'])

⏳ Executing Delimiter Strategy...



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


--- 🧠 تفکرات مدل (برای لاگ سیستم) ---
**مرحله به مرحله فکر کردن (Reasoning):**

1. سن فعلی من ۲۰ سال است.
2. مسئله می‌گوید که سن فعلی من دو برابر سن برادرم است. بنابراین، سن برادرم در حال حاضر برابر است با:
   ۲۰ / ۲ = ۱۰ سال.
3. تفاوت سنی من و برادرم برابر است با:
   ۲۰ - ۱۰ = ۱۰ سال (من ۱۰ سال از برادرم بزرگ‌ترم).
4. مسئله می‌پرسد وقتی من ۴۰ ساله شوم، برادرم چند ساله خواهد بود؟
5. از آنجایی که تفاوت سنی همیشه ثابت می‌ماند، برادرم همیشه ۱۰ سال از من کوچک‌تر خواهد بود.
6. بنابراین، وقتی من ۴۰ ساله شوم، سن برادرم خواهد بود:
   ۴۰ - ۱۰ = ۳۰ سال.

--- ✅ پاسخ نهایی (برای نمایش به کاربر) ---
۳۰


In [4]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# تعریف ساختار داده مطلوب (Schema)
class MathSolution(BaseModel):
    reasoning: str = Field(description="مراحل تفکر و استدلال گام به گام")
    final_answer: float = Field(description="فقط عدد نهایی پاسخ")

# ساخت پارسر هوشمند
parser = PydanticOutputParser(pydantic_object=MathSolution)

# متد .get_format_instructions() دستورالعمل‌های دقیق JSON را می‌سازد
template_pydantic = """
مسئله ریاضی زیر را حل کن.
{format_instructions}

مسئله: {question}
"""

prompt_pydantic = ChatPromptTemplate.from_template(template_pydantic)

# تزریق دستورالعمل‌های ساختار JSON به پرامپت اصلی
final_prompt = prompt_pydantic.partial(format_instructions=parser.get_format_instructions())

# ساخت زنجیره Pydantic
chain_pydantic = final_prompt | model | parser

print("✅ Pydantic Output Parser Chain is ready!")

✅ Pydantic Output Parser Chain is ready!


In [5]:
print("⏳ Executing Pydantic Strategy...\n")

try:
    result = chain_pydantic.invoke({"question": "ریشه دوم عدد ۱۴۴ به علاوه ۵ به توان ۲ چند می‌شود؟"})
    
    # خروجی الان یک آبجکت استاندارد پایتون است!
    print(f"نوع داده خروجی: {type(result)}\n") 
    
    print(f"--- 🧠 استدلال (Reasoning) ---")
    print(result.reasoning)
    
    print(f"\n--- ✅ عدد نهایی (Final Answer) ---")
    print(f"{result.final_answer} (Type: {type(result.final_answer)})")
    
except Exception as e:
    print(f"❌ خطا در پارس کردن: {e}")

⏳ Executing Pydantic Strategy...

نوع داده خروجی: <class '__main__.MathSolution'>

--- 🧠 استدلال (Reasoning) ---
ابتدا ریشه دوم عدد ۱۴۴ را محاسبه می‌کنیم که برابر با ۱۲ است. سپس ۵ به توان ۲ را محاسبه می‌کنیم که برابر با ۲۵ است. در نهایت این دو مقدار را با هم جمع می‌کنیم: ۱۲ + ۲۵ = ۳۷.

--- ✅ عدد نهایی (Final Answer) ---
37.0 (Type: <class 'float'>)
